# Практика 3. Операторы и декораторы

Теория: блоки 6–7.

Девять задач на протоколы операторов и на обёртки функций.

**Как устроено.** Условие → ячейка → проверка. Ячейки, которые надо писать самому, помечены
комментарием в первой строке. Если что-то пошло не так — перезапусти ядро и выполни
служебную ячейку заново.

In [ ]:
# Служебное. Запусти эту ячейку один раз перед задачами.
def check(name, fn):
    """Запускает проверку и печатает результат."""
    try:
        fn()
        print("✓", name)
    except AssertionError as e:
        print("✗", name, "—", e)
    except Exception as e:
        print("✗", name, "—", type(e).__name__, e)


def eq(got, want):
    assert got == want, f"получено {got!r}, ожидалось {want!r}"


def expect(exc, fn):
    try:
        fn()
    except exc:
        return
    raise AssertionError(f"ожидалось {exc.__name__}")

## Задачи

### Задача 1. `__eq__` без `__hash__`

Класс определил только `__eq__`. Предскажи три строки вывода.

In [ ]:
# Ожидаемый вывод:
# True
# TypeError
# None
class M:
    def __init__(self, v):
        self.v = v

    def __eq__(self, other):
        return isinstance(other, M) and self.v == other.v


print(M(1) == M(1))
try:
    {M(1)}
except TypeError:
    print("TypeError")
print(M.__hash__)

### Задача 2. Что печатает `print`

У класса есть и `__str__`, и `__repr__`. Предскажи три строки.

In [ ]:
# Ожидаемый вывод:
# str
# [repr]
# repr
class P:
    def __str__(self):
        return "str"

    def __repr__(self):
        return "repr"


p = P()
print(p)
print([p])
print(f"{p!r}")

### Задача 3. Кто вызывается: `__add__` или `__radd__`

Сложение двух своих объектов и сложение с числом слева. Что напечатается?

In [ ]:
# Ожидаемый вывод:
# __add__
# V(3)
# __radd__
# V(3)
class V:
    def __init__(self, x):
        self.x = x

    def __add__(self, other):
        print("__add__")
        return V(self.x + other.x)

    def __radd__(self, other):
        print("__radd__")
        return V(other + self.x)

    def __repr__(self):
        return f"V({self.x})"


print(V(1) + V(2))
print(1 + V(2))

### Задача 4. Сложение, которое возвращает `None`

`V(1) + 1` не падает с `TypeError`, а молча возвращает `None`. Починить так, чтобы чужой
тип давал `TypeError`, а сложение векторов работало.

In [ ]:
class V:
    def __init__(self, x):
        self.x = x

    def __add__(self, other):
        if isinstance(other, V):
            return V(self.x + other.x)
        return NotImplemented

In [ ]:
# Проверка. Запускай после того, как напишешь решение.
check("сложение векторов", lambda: eq((V(1) + V(2)).x, 3))
check("чужой тип — TypeError", lambda: expect(TypeError, lambda: V(1) + 1))

### Задача 5. Деньги, которые нельзя положить в множество

Класс сравнивает по сумме, но объекты нехешируемы, а сравнение с числом падает
с `AttributeError`. Привести к контракту: равенство по сумме, рабочий `hash`, чужой тип —
`False`, а не исключение.

In [ ]:
class Money:
    def __init__(self, amount):
        self.amount = amount

    def __eq__(self, other):
        return isinstance(other, Money) and self.amount == other.amount

    def __hash__(self):
        return hash(self.amount)

In [ ]:
# Проверка. Запускай после того, как напишешь решение.
check("равенство по сумме", lambda: eq(Money(5) == Money(5), True))
check("в множестве один элемент", lambda: eq(len({Money(5), Money(5)}), 1))
check("чужой тип — False", lambda: eq(Money(5) == 5, False))

### Задача 6. Деньги как значение

**Контракт.** `Money(amount, currency="RUB")`:

- `a + b` складывает только одинаковые валюты, иначе `TypeError`;
- `sum([a, b])` работает (то есть `0 + деньги`);
- равные по сумме и валюте объекты равны и дают одинаковый хеш;
- объекты сортируются по сумме;
- `repr` выглядит как `Money(10, 'RUB')`.

Сигнатура дана, тела — твои.

In [ ]:
class Money:
    def __init__(self, amount, currency="RUB"):
        self.amount = amount
        self.currency = currency

    def __add__(self, other):
        if not isinstance(other, Money):
            return NotImplemented
        if self.currency != other.currency:
            raise TypeError("нельзя складывать разные валюты")
        return Money(self.amount + other.amount, self.currency)

    def __radd__(self, other):
        if other == 0:
            return self
        return NotImplemented

    def __eq__(self, other):
        return (isinstance(other, Money)
                and self.amount == other.amount
                and self.currency == other.currency)

    def __hash__(self):
        return hash((self.amount, self.currency))

    def __lt__(self, other):
        if not isinstance(other, Money):
            return NotImplemented
        return self.amount < other.amount

    def __repr__(self):
        return f"Money({self.amount!r}, {self.currency!r})"

In [ ]:
# Проверка. Запускай после того, как напишешь решение.
a, b = Money(10), Money(5)
check("сложение", lambda: eq((a + b).amount, 15))
check("sum через ноль", lambda: eq(sum([a, b]).amount, 15))
check("равенство и хеш", lambda: eq(len({Money(10), Money(10)}), 1))
check("сортировка по сумме", lambda: eq([m.amount for m in sorted([a, b])], [5, 10]))
check("другая валюта — TypeError", lambda: expect(TypeError, lambda: a + Money(5, "USD")))
check("repr", lambda: eq(repr(Money(10)), "Money(10, 'RUB')"))

### Задача 7. Вектор

**Контракт.** `Vector2D(x, y)`:

- `v + w` и `v - w` дают новый вектор;
- `v * 3` и `3 * v` масштабируют;
- `abs(v)` — длина;
- равные координаты — равные векторы.

Ничего лишнего: ни `__hash__`, ни остальных сравнений.

In [ ]:
from math import hypot


class Vector2D:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __add__(self, other):
        if not isinstance(other, Vector2D):
            return NotImplemented
        return Vector2D(self.x + other.x, self.y + other.y)

    def __sub__(self, other):
        if not isinstance(other, Vector2D):
            return NotImplemented
        return Vector2D(self.x - other.x, self.y - other.y)

    def __mul__(self, scalar):
        if not isinstance(scalar, (int, float)):
            return NotImplemented
        return Vector2D(self.x * scalar, self.y * scalar)

    def __rmul__(self, scalar):
        return self * scalar

    def __abs__(self):
        return hypot(self.x, self.y)

    def __eq__(self, other):
        return (isinstance(other, Vector2D)
                and self.x == other.x
                and self.y == other.y)

In [ ]:
# Проверка. Запускай после того, как напишешь решение.
v = Vector2D(3, 4)
check("сложение", lambda: eq(v + Vector2D(1, 1), Vector2D(4, 5)))
check("вычитание", lambda: eq(v - Vector2D(1, 1), Vector2D(2, 3)))
check("скаляр справа", lambda: eq(v * 2, Vector2D(6, 8)))
check("скаляр слева", lambda: eq(2 * v, Vector2D(6, 8)))
check("длина", lambda: eq(abs(v), 5.0))
check("равенство", lambda: eq(Vector2D(1, 2) == Vector2D(1, 2), True))

### Задача 8. Почему нельзя подменить оператор у экземпляра

`obj.__add__ = lambda other: 42` не влияет на `obj + x`. Объясни, как интерпретатор
ищет метод для оператора и чем это отличается от обычного доступа к атрибуту.

Сформулируй ответ своими словами, без кода и без запуска.

**Ответ.** Для оператора Python ищет специальный метод у типа объекта, а не в словаре экземпляра. Поэтому присваивание `obj.__add__` меняет обычное чтение `obj.__add__`, но не влияет на `obj + x`. Для бинарного оператора Python также может попробовать отражённый метод правого операнда, например `__radd__`.

### Задача 9. Декоратор, который стёр имя функции

Логи и отладчик показывают `wrapper` вместо `ping`. Починить декоратор, не меняя
декорируемую функцию.

In [ ]:
from functools import wraps


def traced(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper


@traced
def ping():
    """Пингует сервис."""
    return "pong"

In [ ]:
# Проверка. Запускай после того, как напишешь решение.
check("имя сохранено", lambda: eq(ping.__name__, "ping"))
check("документация сохранена", lambda: eq(ping.__doc__, "Пингует сервис."))
check("вызов работает", lambda: eq(ping(), "pong"))